# Transaction Foundation Model on Ray — Part 6: Downstream fraud — raw vs FM vs fusion

<div align="left">
  <a target="_blank" href="https://console.anyscale.com/template-preview/fintech_transaction_fm"><img src="https://img.shields.io/badge/🚀 Run_on-Anyscale-9hf"></a>&nbsp;
  <a href="https://github.com/anyscale/templates/tree/main/templates/fintech_transaction_fm" role="button"><img src="https://img.shields.io/static/v1?label=&message=View%20On%20GitHub&color=586069&logo=github&labelColor=2f363d"></a>&nbsp;
</div>

**⏱️ Time to complete**: ~10 min

---

This is the payoff of the whole series: does the foundation-model embedding actually help catch fraud, compared to the features a team already has? We answer it with a controlled experiment. At `small`/`full` the embedding table is millions of rows by hundreds of dimensions — too big to pull onto one node and fit in-process — so the classifier itself trains on the cluster: Ray Data streams the embeddings and `XGBoostTrainer` distributes the boosting across workers (CPU here at `mini`, GPUs at `small`/`full`). It's the same `ScalingConfig` knob the pretrain and embed stages turn.


In [ ]:
import sys, os, json, logging

DEMO_ROOT = os.path.abspath(os.getcwd())
if DEMO_ROOT not in sys.path:
    sys.path.insert(0, DEMO_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ray

from src.paths import artifact_paths, get_demo_base_dir
from src.scale_config import load_scale

SCALE = "mini"                       # same knob as the earlier parts
paths = artifact_paths(get_demo_base_dir(), SCALE)

# How this scale fits the classifier: mini = 1 CPU worker (reproducible, CI-safe);
# small/full = GPU workers. Same code path either way.
ds_cfg = load_scale(SCALE)["downstream"]
NUM_WORKERS, USE_GPU = ds_cfg["num_workers"], ds_cfg["use_gpu"]

# XGBoost reports a metric every boosting round; keep that out of the notebook.
for _n in ("ray.data", "ray.train", "ray.tune"):
    logging.getLogger(_n).setLevel(logging.ERROR)
ray.data.DataContext.get_current().enable_progress_bars = False

## Does the embedding beat the features you already have?

We train the **same XGBoost** on three feature sets, so the only thing that changes is the *representation*:

1. **raw** — the target transaction's own fields (amount, hour, day, MCC): the hand-built baseline a fraud team has today.
2. **fm** — only the FM embedding of the card's history window, with none of the raw fields.
3. **fusion** — the embedding concatenated with the raw fields (Nubank's joint-fusion recipe).

The lift of **fm** and **fusion** over **raw** is the case for a transaction foundation model: it says you can drop or augment a hand-tuned feature pipeline with one pretrained encoder.

The evaluation follows NVIDIA's transaction-FM blueprint so the numbers are comparable: the **temporal 80/10/10 split** from Part 2 (train on the past, test on the most recent 10% — no leakage), per-transaction last-event labels, and **PR-AUC at natural fraud prevalence** (the downsampled normals are importance-weighted back to ~0.1%). An eval fingerprint makes two runs comparable only if they scored the exact same held-out set.

## Fit all three on the cluster

`XGBoostTrainer` is the lesson here. Ray Data reads the embedding Parquet and `expand_features` fans the single `embedding` column out to `emb_0 .. emb_{d-1}` — XGBoost needs a flat numeric matrix — applying the same log-amount transform the raw baseline uses. It runs as a `map_batches`, so the fan-out is distributed and the driver never holds the whole table. Each feature set is then just a **column subset** of that one expanded dataset.

The fit distributes through a single `ScalingConfig`: Ray shards the training rows across workers, each worker builds a `DMatrix` from its shard, and XGBoost's collective runs the boosting. Going from this CPU smoke test to a multi-GPU fit is `num_workers`/`use_gpu` — no code change. The held-out test split is the most-recent 10%, small enough to pull to the driver and score in-process.


In [ ]:
from ray.train import RunConfig, ScalingConfig
from ray.train.xgboost import RayTrainReportCallback, XGBoostTrainer
from src.downstream import expand_features, feature_columns, xgb_params, train_loop_per_worker, evaluate
import time

# Distributed read + fan-out. filter() carves out the temporal splits Part 2 cut.
ds = ray.data.read_parquet(paths["embeddings"]).map_batches(expand_features, batch_format="pandas")
splits = {s: ds.filter(expr=f"split == '{s}'").materialize() for s in ("train", "val", "test")}

# scale_pos_weight (fraud is rare) and the test split we score on the driver.
meta = splits["train"].select_columns(["label", "weight"]).to_pandas()
pos = float(meta["label"].sum())
spw = (len(meta) - pos) / max(pos, 1.0)
test_df = splits["test"].to_pandas()
all_cols = ds.schema().names
print(f"[downstream] loaded  train={len(meta):,}  val={splits['val'].count():,}  test={len(test_df):,}  (emb_dim={len(feature_columns('fm', all_cols))})")

# THE scale knob: 1 CPU worker here, GPU workers at small/full — same code below.
scaling = ScalingConfig(num_workers=NUM_WORKERS, use_gpu=USE_GPU)
# Ray Train persists each fit's checkpoint here; shared storage so any node can read it.
storage = os.path.join(os.path.abspath(paths["downstream"]), "ray_results")
os.makedirs(storage, exist_ok=True)

results, preds = {}, []
for name in ["raw", "fm", "fusion"]:
    cols = feature_columns(name, all_cols)               # raw=4 fields, fm=embedding, fusion=both
    print(f"[downstream] training '{name}'  ({len(cols)} features, num_workers={NUM_WORKERS}, use_gpu={USE_GPU}) ...", flush=True)
    t0 = time.time()
    trainer = XGBoostTrainer(
        train_loop_per_worker=train_loop_per_worker,
        train_loop_config={
            "label_column": "label",
            "params": xgb_params(spw, USE_GPU),          # one recipe, shared across all three
            "num_boost_round": 400,
            "early_stopping_rounds": 30,
        },
        scaling_config=scaling,
        run_config=RunConfig(name=f"downstream_{name}", storage_path=storage),
        datasets={
            "train": splits["train"].select_columns(cols + ["label"]),
            "valid": splits["val"].select_columns(cols + ["label"]),
        },
    )
    booster = RayTrainReportCallback.get_model(trainer.fit().checkpoint)
    results[name], proba = evaluate(test_df, cols, booster)
    print(f"[downstream]   '{name}' done in {time.time() - t0:>4.0f}s  "
          f"AUC-ROC={results[name]['auc_roc']:.4f}  PR-AUC={results[name]['pr_auc']:.4f}", flush=True)
    preds.append(pd.DataFrame({
        "feature_set": name, "label": test_df["label"].to_numpy(),
        "proba": proba, "weight": test_df["weight"].to_numpy(),
    }))
pred = pd.concat(preds, ignore_index=True)

In [ ]:
def _lift(k):
    return results[k]["pr_auc"] - results["raw"]["pr_auc"]

print(f"trained distributed: num_workers={NUM_WORKERS} use_gpu={USE_GPU}")
print(f"test samples: {len(test_df):,}  (natural fraud rate "
      f"{(test_df['weight'] * test_df['label']).sum() / test_df['weight'].sum():.4%})")
print(f"{'feature set':<10}{'AUC-ROC':>10}{'PR-AUC':>10}")
print("-" * 30)
for name in ["raw", "fm", "fusion"]:
    r = results[name]
    print(f"{name:<10}{r['auc_roc']:>10.4f}{r['pr_auc']:>10.4f}")
print("-" * 30)
print(f"FM-only PR-AUC lift vs raw:  {_lift('fm'):+.4f}")
print(f"Fusion PR-AUC lift vs raw:   {_lift('fusion'):+.4f}")

## Reading this honestly at `mini` scale

At `mini` the foundation model trailed the raw baseline — and that's the expected, honest result, not a bug. The encoder here was pretrained for **2 CPU epochs at 64 dimensions, 2 layers**: it has barely learned, so a 64-dim summary of recent history can't yet compete with the target transaction's own raw fields. So `mini` is a smoke test of the *pipeline and the evaluation harness*, not a model you'd ship.

What transfers regardless of scale is the **methodology**: a leakage-free temporal split, metrics reported at the true ~0.1% prevalence (where the published TFMs are measured), and a controlled comparison where representation is the only variable. Run the `small`/`full` GPU pretrain and the picture inverts — `fm` and `fusion` overtake `raw`, matching the lift NVIDIA's blueprint and FATA-Trans report on this dataset. The point of this notebook is the apparatus that will *show* that lift, run honestly at a scale that fits on a CPU.

## The curves

Two views of the same held-out scores, both at natural prevalence. **ROC** shows how well each representation *ranks* fraud above normal — readable here because the `mini` model is weak (at `full` a strong model pushes AUC-ROC toward 1.0, where it saturates and hides differences). **Precision–Recall** shows the operational reality: at ~0.1% fraud, precision collapses, which is exactly why PR-AUC — not accuracy or ROC — is the number a fraud team actually optimizes.

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve, roc_auc_score, average_precision_score

# `pred` was assembled in the training cell above (one row per test sample per
# feature set) — no Parquet round-trip needed inside the notebook.
sns.set_theme(style="white", context="notebook")
COLORS = {"raw": "#4C78A8", "fm": "#F58518", "fusion": "#54A24B"}

fig, (ax_roc, ax_pr) = plt.subplots(1, 2, figsize=(12, 4.8))
for fs in ["raw", "fm", "fusion"]:
    d = pred[pred["feature_set"] == fs]
    fpr, tpr, _ = roc_curve(d["label"], d["proba"], sample_weight=d["weight"])
    auc = roc_auc_score(d["label"], d["proba"], sample_weight=d["weight"])
    ax_roc.plot(fpr, tpr, color=COLORS[fs], lw=1.8, label=f"{fs} (AUC {auc:.3f})")
    prec, rec, _ = precision_recall_curve(d["label"], d["proba"], sample_weight=d["weight"])
    ap = average_precision_score(d["label"], d["proba"], sample_weight=d["weight"])
    ax_pr.plot(rec, prec, color=COLORS[fs], lw=1.8, label=f"{fs} (PR-AUC {ap:.3f})")

ax_roc.plot([0, 1], [0, 1], ls="--", color="#cccccc", lw=1)
ax_roc.set_title("ROC — ranking fraud above normal")
ax_roc.set_xlabel("false positive rate"); ax_roc.set_ylabel("true positive rate"); ax_roc.legend()
ax_pr.set_title("Precision–Recall — at ~0.1% prevalence")
ax_pr.set_xlabel("recall"); ax_pr.set_ylabel("precision"); ax_pr.set_xlim(0, 1); ax_pr.legend()
for ax in (ax_roc, ax_pr):
    sns.despine(ax=ax)
plt.tight_layout()
plt.show()

## Takeaways

- **It scales by config, not by rewrite.** The same `read_parquet → map_batches → XGBoostTrainer` code fits on one CPU worker at `mini` and on `num_workers` GPU workers at `small`/`full` — only `ScalingConfig` changes. Pulling millions of embeddings onto one node and fitting in-process was the bottleneck this stage removes; Ray Data shards the read and the training so the driver never holds the full matrix.
- **The comparison is controlled**: one XGBoost recipe, three feature sets, representation the only variable — so any lift is attributable to the embedding, not to tuning.
- **The protocol is honest and comparable**: temporal split (no leakage), per-transaction labels, PR-AUC at the true ~0.1% prevalence, and an eval fingerprint so two runs are only compared when they scored the same held-out set. It matches NVIDIA's transaction-FM blueprint. (Distributed boosting partitions the data across workers, so exact metric values are reproducible for a given worker count, not bit-identical across counts.)
- **At `mini` the FM trails raw** — a 2-epoch, 64-dim CPU encoder is undertrained. The value of this stage is the apparatus; the lift shows up with the `small`/`full` GPU pretrain, where pretrained TFMs beat hand-built features on this benchmark.

---

## Next

**Part 7 — Online serving with Ray Serve**: deploy the encoder behind an HTTP endpoint that returns an embedding and a fraud score in one call, caching the static card-level fields and autoscaling replicas under load.
